# Day 2: Parsing DataCite Metadata Step by Step

This notebook explains **how to read and parse a DataCite XML metadata record** for a teaching dataset.

## Learning goals
By the end of this notebook, students should be able to:
1. understand what a DataCite metadata record looks like,
2. identify the most important dataset fields,
3. parse the XML file step by step in Python,
4. convert the extracted metadata into a clean table,
5. connect DataCite metadata to the **FAIR principles**.

**Teaching dataset:** `2025 Global Climate Data`


In [1]:
# Notebook bootstrap: download required files from GitHub if missing
from pathlib import Path
from urllib.request import urlretrieve

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/MehrdadJalali-AI/data-management-teaching-materials/main"

required_files = [
    ("data/metadata/climate_dataset_datacite.xml", "data/metadata/climate_dataset_datacite.xml"),
]

for local_rel, github_rel in required_files:
    local_path = Path(local_rel)
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW_BASE}/{github_rel}"
        urlretrieve(url, local_path)
        print(f"Downloaded: {local_path}")
    else:
        print(f"Already available: {local_path}")

print("Bootstrap complete.")

Downloaded: data/metadata/climate_dataset_datacite.xml
Bootstrap complete.


## Step 1 — Import libraries

We use:
- `Path` to work with file paths,
- `xml.etree.ElementTree` to read XML,
- `pandas` to present the results clearly in table form.


In [2]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

In [12]:
def local_name(tag):
    # This function strips the namespace from an XML tag.
    # Example: '{http://datacite.org/schema/kernel-4}resource' becomes 'resource'
    if '}' in tag:
        return tag.split('}', 1)[1]
    return tag

## Step 2 — Load the DataCite XML file

The metadata file is stored locally in the course repository under:

`data/metadata/climate_dataset_datacite.xml`


In [13]:
xml_path = Path("data/metadata/climate_dataset_datacite.xml")

print("XML path:", xml_path)
print("File exists:", xml_path.exists())

if not xml_path.exists():
    raise FileNotFoundError(f"Could not find: {xml_path}")

XML path: data/metadata/climate_dataset_datacite.xml
File exists: True


## Step 3 — Preview the raw XML

Before parsing, it is useful to look at the first part of the XML file.
This helps students see that metadata is stored as **structured tags**.


In [14]:
raw_xml = xml_path.read_text(encoding="utf-8")
print(raw_xml[:1500])

<?xml version="1.0" encoding="UTF-8"?>
<resource
  xmlns="http://datacite.org/schema/kernel-4"
  xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
  xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4/metadata.xsd">

  <identifier identifierType="DOI">10.1234/gcd-2025</identifier>

  <creators>
    <creator>
      <creatorName>Global Climate Data Team</creatorName>
    </creator>
  </creators>

  <titles>
    <title>2025 Global Climate Data</title>
  </titles>

  <publisher>Open Research Repository (Teaching)</publisher>

  <publicationYear>2025</publicationYear>

  <resourceType resourceTypeGeneral="Dataset">Dataset</resourceType>

  <descriptions>
    <description descriptionType="Abstract">
      A small teaching dataset containing fictional annual climate summaries.
      Variables include average temperature, rainfall, and CO2 emissions for two fictional countries
      (Alandia and Borealia) from 2021 to 2023.
    </description>
  

## Step 4 — Parse the XML document

Now we parse the XML into an element tree and inspect the root tag.


In [15]:
tree = ET.parse(xml_path)
root = tree.getroot()

print("Root tag:", root.tag)

Root tag: {http://datacite.org/schema/kernel-4}resource


## Step 5 — Create a simple parser function

We now build a small teaching-oriented parser to extract the most relevant DataCite fields:

- identifier
- creators
- title
- publisher
- publicationYear
- resourceType
- description


In [17]:
def parse_datacite_xml(xml_file: Path) -> dict:
    root = ET.parse(xml_file).getroot()

    identifier = None
    creators = []
    title = None
    publisher = None
    publication_year = None
    resource_type = None
    description = None

    for el in root.iter():
        name = local_name(el.tag)
        text = (el.text or "").strip()

        if name == "identifier" and text and identifier is None:
            identifier = text
        elif name == "creatorName" and text:
            creators.append(text)
        elif name == "title" and text and title is None:
            title = text
        elif name == "publisher" and text and publisher is None:
            publisher = text
        elif name == "publicationYear" and text and publication_year is None:
            publication_year = text
        elif name == "resourceType" and text and resource_type is None:
            resource_type = text
        elif name == "description" and text and description is None:
            description = text

    return {
        "identifier": identifier,
        "creators": creators,
        "title": title,
        "publisher": publisher,
        "publicationYear": publication_year,
        "resourceType": resource_type,
        "description": description,
    }

## Step 6 — Run the parser

Now we apply the parser to the local DataCite XML file.


In [18]:
datacite_meta = parse_datacite_xml(xml_path)
datacite_meta

{'identifier': '10.1234/gcd-2025',
 'creators': ['Global Climate Data Team'],
 'title': '2025 Global Climate Data',
 'publisher': 'Open Research Repository (Teaching)',
 'publicationYear': '2025',
 'resourceType': 'Dataset',
 'description': 'A small teaching dataset containing fictional annual climate summaries.\n      Variables include average temperature, rainfall, and CO2 emissions for two fictional countries\n      (Alandia and Borealia) from 2021 to 2023.'}

In [ ]:
rows = [
    ("identifier", datacite_meta.get("identifier")),
    ("creators", "; ".join(datacite_meta.get("creators", []))),
    ("title", datacite_meta.get("title")),
    ("publisher", datacite_meta.get("publisher")),
    ("publicationYear", datacite_meta.get("publicationYear")),
    ("resourceType", datacite_meta.get("resourceType")),
    ("description", datacite_meta.get("description")),
]

table = pd.DataFrame(rows, columns=["field", "value"])
table

,field,value
0,identifier,10.1234/gcd-2025
1,creators,Global Climate Data Team
2,title,2025 Global Climate Data
3,publisher,Open Research Repository (Teaching)
4,publicationYear,2025
5,resourceType,Dataset
6,description,A small teaching dataset containing fictional ...
